# Gemini 3.8 Flash coverage audit · 1.4.2

Executed checks of source conditions, likelihood units, unchanged modeling choices, publication coverage and accepted numerics. Capture timestamps use UTC; the local review date is 6 September 2026 in Moscow. This evidence refresh does not claim an independently validated improvement to the capability formula. Raw source pages and private task contents are not redistributed.


In [1]:
import json, math, hashlib
from pathlib import Path
HERE = next(p for p in [Path.cwd(), Path.cwd()/"docs/audits/1.4.2-gemini-coverage"] if (p/"audit-summary.json").exists())
def read(name): return json.loads((HERE/name).read_text())
a = read("audit-summary.json")
p = read("accepted-input.json")
d = read("accepted-diagnostics.json")
focal = read("focal-prepared-observations.json")
assert hashlib.sha256((HERE/"accepted-input.json").read_bytes()).hexdigest() == a["accepted_input_sha256"]
print("Pinned accepted input:", a["accepted_input_sha256"])


Pinned accepted input: e640a1c41d5c4bdb1c80e4ad93db208cce50d099ad9308a5fa174a7fd0d647c9


In [2]:
before, after = a["before"]["gemini"], a["after"]["gemini"]
assert (before["distinct_conditions"], after["distinct_conditions"]) == (9,18)
assert (before["indexes"]["mixed"]["fitted_cells"], after["indexes"]["mixed"]["fitted_cells"]) == (3,4)
assert len(set(r["benchmarkId"] for r in focal)) == 5
assert a["before"]["catalog_models"] == a["after"]["catalog_models"] == 126
print("Distinct reported conditions:", before["distinct_conditions"], "→", after["distinct_conditions"])
print("Fitted conditions across configurations: 3 → 5; maximum-profile direct cells: 3 → 4; source rows:", before["source_rows"], "→", after["source_rows"])
print("Coverage counts refer to public-source evidence, excluding AA display-only overlays.")


Distinct reported conditions: 9 → 18
Fitted conditions across configurations: 3 → 5; maximum-profile direct cells: 3 → 4; source rows: 12 → 20
Coverage counts refer to public-source evidence, excluding AA display-only overlays.


In [3]:
finance = read("native-finance-observations.json")
g = next(r for r in finance if r["modelSnapshotId"] == "gemini-3.8-flash")
assert len(finance) == 38 and all(r["likelihood"] == "a_prime" for r in finance)
assert g["score"] == 61.435 and g["standardError"] == .128 and g["nRuns"] == 3 and g["nTasks"] == 450
expected = (.00128 / (.61435 * (1-.61435)))**2
assert math.isclose(g["variance"], expected, rel_tol=1e-12)
assert g["metadataIncomplete"] and g["effortTier"] == "high"
assert not any(r["benchmarkId"].endswith("all-pass") for r in focal)
print("Finance:", len(finance), "eligible configurations; fraction-to-logit SEM variance", expected)
print("Run SEM is repeatability on fixed tasks, not new-task generalization. Learned discrepancy remains enabled.")


Finance: 38 eligible configurations; fraction-to-logit SEM variance 2.9187812161261038e-05
Run SEM is repeatability on fixed tasks, not new-task generalization. Learned discrepancy remains enabled.


In [4]:
old = json.loads((HERE.parent/"1.4.1-coverage/accepted-input.json").read_text())
for key in ["trait_structure", "calibration_panel_system_ids", "profiles", "priors", "inference", "metadata_incomplete_multiplier"]:
    assert old[key] == p[key], key
assert p["n_benchmarks"] == 19 and len(p["observations"]) == 921
assert d["divergences"] == 0 and d["posterior_draws"] == 12000
assert max(x["rhat"] for x in d["parameters"].values()) <= 1.01
assert min(min(x["ess_bulk"],x["ess_tail"]) for x in d["parameters"].values()) >= 400
assert min(d["ebfmi"]) >= .3 and max(d["mcse_display_points"].values()) <= .3
print("Existing formula, panel, profiles, priors and numerical settings unchanged; acceptance checks passed.")
print(json.dumps(a["fit"], indent=2))


Existing formula, panel, profiles, priors and numerical settings unchanged; acceptance checks passed.
{
  "observations": 921,
  "models": 110,
  "systems": 131,
  "benchmark_conditions": 19,
  "divergences": 0,
  "retained_draws": 12000,
  "max_rhat": 1.0033043795720782,
  "min_ess": 1822.9626046755293,
  "min_ebfmi": 0.8675545836219911,
  "max_mcse": 0.29885668454607855
}


In [5]:
for kind in ["mixed","agentic","chat"]:
    b,c = before["indexes"][kind], after["indexes"][kind]
    print(kind, "median", round(b["median"],2), "→", round(c["median"],2), "90% interval", c["ci90"])
assert all(r["ci90"][0] <= r["median"] <= r["ci90"][1] for r in after["indexes"].values())
print("Changed scores reflect corrected/added evidence and a refit; no target ordering was imposed.")


mixed median 56.32 → 61.79 90% interval [56.667854, 67.75362]
agentic median 57.98 → 61.41 90% interval [56.114437, 68.07289]
chat median 54.47 → 60.69 90% interval [54.065697, 67.84733]
Changed scores reflect corrected/added evidence and a refit; no target ordering was imposed.


## Assessment

Ready with stated limitations: all new factual values have primary sources, admitted observations preserve reported uncertainty, and the fitted model passes the existing numerical gates. LVBench tool documentation conflicts, absent per-run Finance values, unpinned grader revisions, unavailable Harvey uncertainty units and unmatched source model releases remain explicit. More data does not establish predictive accuracy on unseen evaluations. The source and integration reviews describe all retained gaps.
